# IoT / MQTT Startup Template
### paho-mqtt (Callback API v2)

Copy this notebook as the starting point for MQTT publish/subscribe work
against a broker (e.g. for talking to an ESP32/STM32 device over MQTT).

**Important: this notebook only works in local Jupyter/Spyder, NOT in
JupyterLite.** MQTT needs a real outbound TCP socket to reach a broker --
browser sandboxing blocks that entirely, the same way it blocks direct
serial-port access. There is no browser-compatible fallback for this.

**API note:** `paho-mqtt` 2.x changed the callback signatures and now
requires explicitly specifying the callback API version. This template
uses `CallbackAPIVersion.VERSION2` (the current one) -- if you copy
snippets from older tutorials online, they're very likely written for
the deprecated v1 API and will need updating.


## Setup

In [ ]:
import paho.mqtt.client as mqtt
import time
import json


## Broker connection settings

Point this at your actual broker. `test.mosquitto.org` is a public
broker that's fine for quick experiments -- don't rely on it for
anything real (no auth, no privacy, occasionally offline).


In [ ]:
BROKER_HOST = "test.mosquitto.org"
BROKER_PORT = 1883
TOPIC = "auralius/test/temperature"   # change to your own topic


## Subscriber client

Define callbacks, connect, and start listening. `loop_start()` runs
the network loop in a background thread so the rest of the notebook
stays usable while messages arrive.


In [ ]:
def on_connect(client, userdata, connect_flags, reason_code, properties):
    if reason_code == 0:
        print(f"Connected OK, subscribing to '{TOPIC}'")
        client.subscribe(TOPIC)
    else:
        print(f"Connection failed: {reason_code}")


def on_message(client, userdata, msg):
    payload = msg.payload.decode(errors="replace")
    print(f"[{msg.topic}] {payload}")


sub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
sub_client.on_connect = on_connect
sub_client.on_message = on_message

sub_client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
sub_client.loop_start()   # background thread -- non-blocking


## Publisher: send a test message

Run this cell any time to publish -- the subscriber above (if still
running) should print it via `on_message`.


In [ ]:
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)

message = json.dumps({"value": 23.5, "unit": "C", "ts": time.time()})
pub_client.publish(TOPIC, message)

pub_client.disconnect()
print("Published:", message)


## Stop the subscriber

Always clean up the background loop and disconnect when you're done --
an orphaned `loop_start()` thread will keep running in the kernel
otherwise.


In [ ]:
sub_client.loop_stop()
sub_client.disconnect()
print("Subscriber stopped.")


## Alternative: blocking loop instead of a background thread

Useful for a simple script-style logger where you don't need the
notebook interactive while listening -- this blocks the cell until you
interrupt the kernel (Stop button / Ctrl+C equivalent).


In [ ]:
# sub_client2 = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
# sub_client2.on_connect = on_connect
# sub_client2.on_message = on_message
# sub_client2.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
# sub_client2.loop_forever()   # blocks here until interrupted


## Scratch space

Everything below is just working room -- delete/replace freely.
